# Glioma Segmentation — Colab GPU Demo

This notebook runs a lightweight, single-process version of the on-premise glioma segmentation service on a Google Colab GPU runtime.

**Use case:** quick demos, debugging, and viewer testing.
**Not for:** clinical production (no persistence, no auth, session expires).

## 1. Check GPU runtime
Make sure you selected a GPU runtime: `Runtime → Change runtime type → Hardware accelerator → GPU`.

In [ ]:
!nvidia-smi

## 2. Get the code
Replace the URL below with your own GitHub repository, or upload the project folder manually and skip this cell.

In [ ]:
# Replace with your repository URL
!git clone https://github.com/YOUR_USERNAME/AdultGliomaSegmentation.git
%cd AdultGliomaSegmentation

## 3. Install CUDA-enabled PyTorch and project dependencies

In [ ]:
# CUDA 12.1 wheels for Colab GPUs
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install the project package and its dependencies
!pip install -q -e .

## 4. Upload model weights
Option A: copy from Google Drive (recommended if you already stored `models_registry` there).
Option B: upload the `models_registry` folder manually using the Files panel on the left.

In [ ]:
# Option A example: mount Google Drive and copy the registry
from google.colab import drive
drive.mount('/content/drive')
!rm -rf models_registry
!cp -r /content/drive/MyDrive/models_registry .

In [ ]:
# Verify the model config is present
!ls models_registry/v1.0.0/

## 5. Start the API server
This loads the ensemble once at startup, so inference requests are fast.

In [ ]:
import os
import subprocess
import sys
import time

os.environ['GLIOMA__MODEL_VERSION'] = 'v1.0.0'

# Start the lightweight Colab app in the background
proc = subprocess.Popen(
    [sys.executable, '-m', 'app.main_colab'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Wait for the ensemble to load
time.sleep(60)
print('Server PID:', proc.pid)

## 6. Expose the server with ngrok
1. Get a free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
2. Replace `YOUR_NGROK_TOKEN` below.

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token('YOUR_NGROK_TOKEN')
public_url = ngrok.connect(8500, 'http')
print('Public URL:', public_url)
print('Open viewer at:', f'{public_url}/viewer/app.html')

## 7. Test with a sample scan
Upload a NIfTI file and inspect the result.

In [ ]:
import requests

url = f'{public_url}/api/v1/segmentation/upload'
with open('data/dev/test_scan_4ch_128.nii.gz', 'rb') as f:
    r = requests.post(url, files={'file': f})
print(r.status_code)
print(r.json())

In [ ]:
import time

for _ in range(30):
    r = requests.get(f'{public_url}/api/v1/segmentation/result/1')
    data = r.json()
    print(data['status'], data.get('message', ''))
    if data['status'] == 'success':
        print(data)
        break
    time.sleep(5)

## 8. Stop the tunnel and server (when finished)

In [ ]:
ngrok.kill()
proc.terminate()